# Component 2 - Demand and Spam Annotation Preparation

This notebook prepares a deterministic manual annotation sample for audience-demand labels and separate human spam-status labels. It does not train a classifier, run BERTopic, calculate Jensen-Shannon Divergence, collect new YouTube data, or connect to Supabase.

Visible notebook outputs are limited to aggregate checks and distribution tables; raw comment text is written only to the local annotation CSV used for manual labelling.

In [1]:
from pathlib import Path
import sys

import pandas as pd

start = Path.cwd().resolve()
for candidate in [start, *start.parents]:
    if (candidate / "research" / "component_2" / "src" / "annotation.py").exists():
        REPO_ROOT = candidate
        break
else:
    raise RuntimeError("Could not locate repository root from notebook working directory")

SRC = REPO_ROOT / "research" / "component_2" / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from annotation import (
    ANNOTATION_COLUMNS,
    SPAM_ANNOTATION_COLUMNS,
    AnnotationSamplingConfig,
    add_spam_annotation_fields,
    create_annotation_sample,
    create_annotation_template,
    ensure_no_human_annotations,
    nonblank_annotation_counts,
    prepare_annotation_source,
    validate_existing_annotation_template,
    write_annotation_outputs,
)

PROCESSED_DIR = REPO_ROOT / "data" / "processed" / "component_2"
ANNOTATION_DIR = PROCESSED_DIR / "annotation"
ANNOTATION_CSV = ANNOTATION_DIR / "component2_demand_annotation_sample.csv"

preprocessed_path = PROCESSED_DIR / "preprocessed_comments.csv"
validated_path = PROCESSED_DIR / "validated_comments.csv"

assert preprocessed_path.exists(), f"Missing {preprocessed_path}"
assert validated_path.exists(), f"Missing {validated_path}"

## Load Phase 1 Outputs

The annotation source is built from Phase 1 preprocessed comments plus the validated comment metadata needed for channel names, reply status, and publication time.

In [2]:
preprocessed_comments = pd.read_csv(
    preprocessed_path,
    dtype={"comment_id": "string", "channel_id": "string", "video_id": "string"},
)
validated_comments = pd.read_csv(
    validated_path,
    dtype={"comment_id": "string", "parent_comment_id": "string"},
)

source = prepare_annotation_source(preprocessed_comments, validated_comments)

source_summary = pd.DataFrame(
    [
        {
            "source_comments": len(source),
            "source_channels": source["channel_id"].nunique(),
            "source_videos": source["video_id"].nunique(),
            "reply_comments": int(source["is_reply"].sum()),
            "empty_cleaned_text": int(source["cleaned_text"].fillna("").str.strip().eq("").sum()),
        }
    ]
)
source_summary

,source_comments,source_channels,source_videos,reply_comments,empty_cleaned_text
0,2000,5,255,367,0


## Create or Safely Extend Deterministic Annotation Sample

The target sample size remains 300 comments. The sampler balances channels first, then deliberately includes minority and review-sensitive cases such as mixed-script comments, emoji-only comments, replies, very short comments, Sinhala-script comments, and Latin-script comments. Selection order is deterministic using a stable hash of `comment_id` and the configured seed.

If the annotation CSV already exists, this notebook validates that the same comment rows and ordering are preserved. It stops before writing if any human annotation field is nonblank.

In [3]:
config = AnnotationSamplingConfig(sample_size=300, random_seed=334)
sample = create_annotation_sample(source, config)
fresh_template = create_annotation_template(sample)

if ANNOTATION_CSV.exists():
    existing_template = pd.read_csv(ANNOTATION_CSV, dtype="string").fillna("")
    validate_existing_annotation_template(existing_template, fresh_template)
    ensure_no_human_annotations(existing_template)
    template = add_spam_annotation_fields(existing_template)
    template_action = "extended_existing_unlabelled_csv"
else:
    template = fresh_template
    template_action = "created_new_csv"

paths = write_annotation_outputs(template, ANNOTATION_DIR)

annotation_counts = nonblank_annotation_counts(template)
checks = pd.DataFrame(
    [
        {
            "template_action": template_action,
            "annotation_rows": len(template),
            "unique_comment_ids": template["comment_id"].nunique(),
            "channels_covered": template["channel_id"].nunique(),
            "videos_covered": template["video_id"].nunique(),
            "blank_primary_labels": int(template["primary_label"].fillna("").eq("").sum()),
            "blank_demand_subtypes": int(template["demand_subtype"].fillna("").eq("").sum()),
            "blank_spam_status_fields": int(
                sum(template[column].fillna("").eq("").sum() for column in SPAM_ANNOTATION_COLUMNS)
            ),
            "human_annotation_nonblank_total": sum(annotation_counts.values()),
            "schema_matches_expected": template.columns.tolist() == ANNOTATION_COLUMNS,
        }
    ]
)
checks

,template_action,annotation_rows,unique_comment_ids,channels_covered,videos_covered,blank_primary_labels,blank_demand_subtypes,blank_spam_status_fields,human_annotation_nonblank_total,schema_matches_expected
0,extended_existing_unlabelled_csv,300,300,5,152,300,300,1200,0,True


## Sample Distribution Checks

These aggregate tables verify coverage without displaying raw comment text.

In [4]:
by_channel = pd.read_csv(paths["by_channel"])
by_script_category = pd.read_csv(paths["by_script_category"])
by_sampling_stratum = pd.read_csv(paths["by_sampling_stratum"])
by_reply_status = pd.read_csv(paths["by_reply_status"])

by_channel

,channel_id,channel_name,comments,videos
0,UCCK3OZi788Ok44K97WAhLKQ,Ada Derana,60,39
1,UCHKtXGbqRIM0hA4Manb9DNg,Ada Derana Sports,60,27
2,UCOPPH1_whpThgY6e7r3qg-w,FM Derana,60,25
3,UCRDDfbYPHX_GUJ4lcQYTc8A,TV Derana,60,26
4,UCs8gVM6lPhRZk3L_zZmedpw,Dream Star,60,35


In [5]:
by_script_category

,script_category,count
0,latin_script,106
1,sinhala_script,91
2,emoji_only,61
3,mixed_sinhala_latin,40
4,other_or_symbolic,2


In [6]:
by_sampling_stratum

,sampling_stratum,count
0,sinhala_script,74
1,latin_script,68
2,emoji_only,61
3,reply,43
4,mixed_sinhala_latin,40
5,very_short,14


In [7]:
by_reply_status

,is_reply,count
0,False,234
1,True,66


## Output Files

The generated annotation CSV contains public comment text and should stay local unless the team explicitly approves a sharing path. It is ignored by Git under `data/processed/component_2/annotation/`. The spam columns are blank human annotation fields and are separate from the demand label fields.

In [8]:
pd.DataFrame(
    [{"artifact": name, "path": str(path.relative_to(REPO_ROOT))} for name, path in paths.items()]
)

,artifact,path
0,annotation_csv,data/processed/component_2/annotation/componen...
1,by_channel,data/processed/component_2/annotation/by_chann...
2,by_script_category,data/processed/component_2/annotation/by_scrip...
3,by_sampling_stratum,data/processed/component_2/annotation/by_sampl...
4,by_reply_status,data/processed/component_2/annotation/by_reply...


## Manual Labelling Instructions

Use `research/component_2/annotation_guidelines.md` while filling the blank demand and spam annotation fields. Do not create synthetic labels. Demand classification and spam classification are separate decisions. After human annotation is complete, the next research step is inter-annotator/adjudication review before any classifier, topic modelling, automated spam filtering, or demand-supply divergence work begins.